# 02 - Tr?na och inferera

Det h?r notebooket bygger en riktig ML-pipeline med preprocessing, KMeans-clustering, modellval, sparad pipeline och inference p? nya kunder.

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
if not (ROOT / 'data').exists() and (ROOT.parent.parent / 'data').exists():
    ROOT = ROOT.parent.parent

In [ ]:
data = pd.read_csv(ROOT / 'data' / 'processed' / 'customer_enriched.csv', parse_dates=['last_purchase_date'])
data.head()

In [ ]:
feature_cols_num = ['recency_days', 'frequency', 'monetary', 'avg_order_value', 'basket_size', 'avg_items_per_invoice', 'unique_products', 'avg_unit_price']
feature_cols_cat = ['RegionGroup']
X_df = data[feature_cols_num + feature_cols_cat].copy()
X_df.head()

In [ ]:
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log1p', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
    ('scaler', StandardScaler()),
])

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, feature_cols_num),
    ('cat', categorical_pipe, feature_cols_cat),
])

X = preprocessor.fit_transform(X_df)
X.shape

In [ ]:
selection_rows = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X)
    selection_rows.append({
        'k': k,
        'silhouette_score': silhouette_score(X, labels),
        'inertia': km.inertia_,
    })
scores = pd.DataFrame(selection_rows)
scores

In [ ]:
sns.set_theme(style='whitegrid', context='talk')
plt.figure(figsize=(10, 6))
sns.barplot(data=scores, x='k', y='silhouette_score', color='#9ecae1')
sns.lineplot(data=scores, x='k', y='silhouette_score', color='#08519c', marker='o')
plt.title('Silhouette score by k')
plt.show()

In [ ]:
best_k = 4
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('kmeans', KMeans(n_clusters=best_k, random_state=42, n_init=20)),
])

clusters = pipeline.fit_predict(X_df)
data['Cluster'] = clusters
data[['CustomerID', 'RegionGroup', 'recency_days', 'frequency', 'monetary', 'Cluster']].head()

In [ ]:
profile = data.groupby('Cluster').agg(
    CustomerCount=('CustomerID', 'count'),
    MeanRecency=('recency_days', 'mean'),
    MeanFrequency=('frequency', 'mean'),
    MeanMonetary=('monetary', 'mean'),
    MeanAOV=('avg_order_value', 'mean'),
    MeanBasketSize=('basket_size', 'mean'),
    MeanUniqueProducts=('unique_products', 'mean'),
).round(1)
profile

In [ ]:
champions = profile.sort_values(['MeanRecency', 'MeanMonetary']).index[0]
at_risk = profile.sort_values(['MeanRecency', 'MeanMonetary'], ascending=[False, True]).index[0]
big_ticket = profile.sort_values('MeanAOV', ascending=False).index[0]
remaining = [c for c in profile.index if c not in {champions, at_risk, big_ticket}]
regulars = remaining[0] if remaining else champions

cluster_name_map = {
    champions: 'Champions',
    big_ticket: 'Big-ticket occasional',
    regulars: 'Regulars',
    at_risk: 'At-risk low value',
}
if len(set(cluster_name_map.values())) < 4 and len(profile.index) >= 4:
    ordered = profile.sort_values('MeanMonetary', ascending=False).index.tolist()
    cluster_name_map = {
        ordered[0]: 'Champions',
        ordered[1]: 'Big-ticket occasional',
        ordered[2]: 'Regulars',
        ordered[3]: 'At-risk low value',
    }

data['ClusterName'] = data['Cluster'].map(cluster_name_map)

joblib.dump(pipeline, ROOT / 'models' / 'clustering_pipeline.joblib')
data.to_csv(ROOT / 'outputs' / 'clustered_customers.csv', index=False)
scores.to_csv(ROOT / 'outputs' / 'model_selection.csv', index=False)
profile.to_csv(ROOT / 'outputs' / 'cluster_profile.csv')
cluster_name_map

In [ ]:
cluster_summary = data.groupby('ClusterName').agg(
    customers=('CustomerID', 'count'),
    mean_recency=('recency_days', 'mean'),
    mean_monetary=('monetary', 'mean'),
    mean_frequency=('frequency', 'mean'),
).round(1)
cluster_summary

In [ ]:
new_customers = pd.read_csv(ROOT / 'data' / 'raw' / 'new_customers.csv')
regions = pd.read_csv(ROOT / 'data' / 'regions.csv')
new_enriched = new_customers.merge(regions[['Country', 'RegionGroup']], on='Country', how='left')
new_predictions = pipeline.predict(new_enriched[feature_cols_num + feature_cols_cat])
new_enriched['PredictedCluster'] = new_predictions
new_enriched['PredictedClusterName'] = new_enriched['PredictedCluster'].map(cluster_name_map)
new_enriched.to_csv(ROOT / 'outputs' / 'inference_results.csv', index=False)
new_enriched[['CustomerID', 'Country', 'PredictedClusterName']]

The fitted preprocessing and clustering steps are saved together in `models/clustering_pipeline.joblib`, so inference uses the exact same transformations as training.